In [516]:
import pandas as pd
import locale

In [517]:
eurocopa_fem_stats_2001 = pd.read_csv("datasetsWC26/datasetsWC26/2001eurocopa_fem_stats.csv", sep=",")
match_lineups_2026 = pd.read_csv("datasetsWC26/datasetsWC26/2026match_lineups.csv", sep=";")
match_register_2026 = pd.read_csv("datasetsWC26/datasetsWC26/2026match_register.csv", sep=";")
nationalteams_2026 = pd.read_csv("datasetsWC26/datasetsWC26/2026nationalteams.csv", sep=",")
player_stats_2026 = pd.read_csv("datasetsWC26/datasetsWC26/2026player_stats.csv", sep=";")
squads_players_2026 = pd.read_csv("datasetsWC26/datasetsWC26/2026squads_and_players.csv", sep=";")
index_referees = pd.read_csv("datasetsWC26/datasetsWC26/index_referees.csv", sep=";")
matches_mundial202 = pd.read_csv("datasetsWC26/datasetsWC26/matches_mundial202.csv", sep=",")
matches_mundial2026 = pd.read_csv("datasetsWC26/datasetsWC26/matches_mundial2026.csv", sep=";")
tournament_stages = pd.read_csv("datasetsWC26/datasetsWC26/tournament_stages.csv", sep=",")
worldcup_venues = pd.read_csv("datasetsWC26/datasetsWC26/worldcup_venues.csv", sep=";")


lista_dfs = [
    ("2001eurocopa_fem_stats", eurocopa_fem_stats_2001),
    ("2026match_lineups", match_lineups_2026),
    ("2026match_register", match_register_2026),
    ("2026nationalteams", nationalteams_2026),
    ("2026player_stats", player_stats_2026),
    ("2026squads_and_players", squads_players_2026),
    ("index_referees", index_referees),
    ("matches_mundial202", matches_mundial202),
    ("matches_mundial2026", matches_mundial2026),
    ("tournament_stages", tournament_stages),
    ("worldcup_venues", worldcup_venues),
]

In [518]:
def resumen(df: pd.DataFrame, nombre=""):
    print('\n   '  ,nombre)
    print(f'Tamaño: {df.shape[0]} filas x {df.shape[1]} columnas\n')
    print('Columnas y tipos:')
    print(df.dtypes)
    print('\nValores nulos:')
    print(df.isnull().sum())
    print('\n','-----'*30)


for nombre, df in lista_dfs:
    resumen(df, nombre)


    2001eurocopa_fem_stats
Tamaño: 19 filas x 47 columnas

Columnas y tipos:
id_match                   int64
home_team                 object
away_team                 object
home_team_code            object
away_team_code            object
home_score               float64
away_score                object
home_penalty             float64
away_penalty             float64
home_score_total         float64
away_score_total         float64
winner                    object
winner_reason             object
year                       int64
date                      object
date_time                 object
utc_offset_hours         float64
group_name                object
matchday_name            float64
condition_humidity       float64
condition_pitch          float64
condition_temperature    float64
condition_weather        float64
condition_wind_speed     float64
status                    object
type                      object
round                     object
round_mode                objec

In [519]:
partidos = pd.DataFrame()
eventos = pd.DataFrame()


#index_partido
matches_mundial2026 = matches_mundial2026.drop_duplicates().copy()
partidos['index_partido'] = matches_mundial2026['match_id']

#Fecha
#esto lo he tenido que buscar ya que daba error con las fechas en español
s = matches_mundial2026['date'].astype(str).str.strip()
# 2. Traducir meses en español a inglés
meses_es_en = {
    r'\benero\b': 'January', r'\bfebrero\b': 'February', r'\bmarzo\b': 'March',
    r'\babril\b': 'April', r'\bmayo\b': 'May', r'\bjunio\b': 'June',
    r'\bjulio\b': 'July', r'\bagosto\b': 'August', r'\bseptiembre\b': 'September',
    r'\boctubre\b': 'October', r'\bnoviembre\b': 'November', r'\bdiciembre\b': 'December',
    r'\b de \b': ' '
}
for es, en in meses_es_en.items():
    s = s.str.replace(es, en, regex=True, case=False)
# 3. Quitar la 'Z' al final de las marcas de tiempo ISO (evita problemas de zona horaria)
s = s.str.replace('Z$', '', regex=True)
# 4. Convertir a datetime con Pandas
matches_mundial2026['date'] = pd.to_datetime(s, format='mixed', errors='coerce')

partidos['fecha_partido'] = matches_mundial2026['date']


#ronda
tournament_stages = tournament_stages.drop_duplicates().copy()
matches_mundial2026['index_partido'] = matches_mundial2026['match_id']
matches_mundial2026 = matches_mundial2026.merge(tournament_stages[['stage_name', 'stage_id']],how = 'left', on='stage_id' )
partidos = partidos.merge(matches_mundial2026[['stage_name', 'index_partido']],how = 'left', on='index_partido')
partidos = partidos.rename(columns={'stage_name': 'ronda'})

#estadio
worldcup_venues = worldcup_venues.drop_duplicates().copy()
matches_mundial2026 = matches_mundial2026.merge(worldcup_venues[['stadium_name', 'venue_id', 'city', 'country']],how = 'left', on='venue_id' )
partidos = partidos.merge(matches_mundial2026[['stadium_name', 'index_partido', 'city', 'country']],how = 'left', on='index_partido')
partidos = partidos.rename(columns={'stadium_name': 'estadio'})

#ciudad_partido
partidos = partidos.rename(columns={'city': 'ciudad_partido'})

#pais_partido
partidos = partidos.rename(columns={'country': 'pais_partido'})
partidos['pais_partido'] = partidos['pais_partido'].str.upper()

#seleccion_local
nationalteams_2026 = nationalteams_2026.drop_duplicates().copy()
matches_mundial2026 = matches_mundial2026.merge(
    nationalteams_2026[['team_id','team_name']], 
    left_on='home_team_id', 
    right_on='team_id', 
    how='left'
)
matches_mundial2026 = matches_mundial2026.rename(columns={'team_name': 'seleccion_local'})
partidos = partidos.merge(matches_mundial2026[['index_partido', 'seleccion_local']],how = 'left', on='index_partido')


#seleccion_visitante
matches_mundial2026 = matches_mundial2026.merge(
    nationalteams_2026[['team_id','team_name']], 
    left_on='away_team_id', 
    right_on='team_id', 
    how='left'
)

partidos = partidos.merge(matches_mundial2026[['index_partido', 'team_name']],how = 'left', on='index_partido')
partidos = partidos.rename(columns={'team_name': 'seleccion_visitante'})

#marcador_local
partidos = partidos.merge(matches_mundial2026[['index_partido', 'home_score', 'away_score', 'home_penalty_score', 'away_penalty_score', 'result_type']],how = 'left', on='index_partido')
partidos = partidos.rename(columns={'home_score': 'marcador_local'})

#marcador_visitante
partidos = partidos.rename(columns={'away_score': 'marcador_visitante'})

#prorroga
partidos = partidos.rename(columns={'result_type': 'prorroga'})
partidos['prorroga'] = partidos['prorroga'].map({'Regular':'NO','AET':'SI','Penalties':'SI'})

#penalty_local
partidos = partidos.rename(columns={'home_penalty_score': 'penalty_local'})
partidos['penalty_local'] = (partidos['penalty_local'].fillna(0)/10).astype(int)

#penalty_visitante
partidos = partidos.rename(columns={'away_penalty_score': 'penalty_visitante'})
partidos['penalty_visitante'] = (partidos['penalty_visitante'].fillna(0)/10).astype(int)

#arbitro_partido
index_referees = index_referees.drop_duplicates().copy()
matches_mundial2026 = matches_mundial2026.merge(index_referees[['referee_id', 'name']], how='left', on='referee_id')
partidos = partidos.merge(matches_mundial2026[['index_partido', 'name']], how='left', on='index_partido')
partidos = partidos.rename(columns={'name': 'arbitro_partido'})


In [ ]:
jugadores = pd.DataFrame()
#index inicial, ya que hay jugadores que comparten nombre (Ali Ahmed)
jugadores['player_id'] = player_stats_2026['player_id']

#nombre_jugador
jugadores['nombre_jugador'] = player_stats_2026['player_name']

#seleccion_nacional
nationalteams_2026 = nationalteams_2026.drop_duplicates().copy()
player_stats_2026 = player_stats_2026.merge(nationalteams_2026[['team_id', 'team_name']], how='left', on='team_id')
jugadores['seleccion_nacional'] = player_stats_2026['team_name']

#club_jugador
jugadores = jugadores.merge(squads_players_2026[['player_id', 'club_team']], on='player_id', how='left')
jugadores = jugadores.rename(columns={'club_team':'club_jugador'})

#partidos_seleccion
jugadores = jugadores.merge(squads_players_2026[['player_id', 'caps']], on='player_id', how='left')
jugadores = jugadores.rename(columns={'caps':'partidos_seleccion'})


#posicion
jugadores['posicion'] = player_stats_2026['position']

#valor_mercado-------------(no terminado, queda mucho que limpiar y NaN que sustituir)-------------------------
jugadores = jugadores.merge(squads_players_2026[['player_id', 'market_value_eur']], on='player_id', how='left')
jugadores = jugadores.rename(columns={'market_value_eur':'valor_mercado'})
jugadores.groupby(jugadores['club_jugador'])['valor_mercado'].mean()

#partidos_jugados
jugadores['partidos_jugados'] = player_stats_2026['matches_played']

#minutos_jugados
jugadores['minutos_jugados'] = player_stats_2026['minutes_played']

#goles
jugadores['goles'] = player_stats_2026['goals'].fillna(0).astype(int)

#asistencias
jugadores['asistencias'] = player_stats_2026['assists'].fillna(0).astype(int)

#tiros
jugadores['tiros'] = player_stats_2026['shots'].fillna(0).astype(int)

#tarjetas_amarillas
jugadores['tarjetas_amarillas'] = player_stats_2026['yellow_cards'].fillna(0).astype(int)

#tarjetas_rojas
jugadores['tarjetas_rojas'] = player_stats_2026['red_cards'].fillna(0).astype(int)

#paradas
jugadores['paradas'] = player_stats_2026['saves'].fillna(0).astype(int)

#goles_concedidos
jugadores['goles_concedidos'] = player_stats_2026['goals_conceded'].fillna(0).astype(int)

#calificacion_media
jugadores['calificacion_media'] = player_stats_2026['average_rating'].fillna(5)

